In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import annoy

# Setup paths
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.models.VAE import ConvVAE, load_checkpoint as load_vae_checkpoint
from src.models.resnet import build_resnet_classifier
from src.smoothing.workflow import fit_local_pca, whiten, unwhiten
from src.certify.randomized import certify_token_from_counts

# Reproducibility
SEED = 73
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Repo root: {repo_root}")

## Configuration

Set paths for your datasets and models. Adjust based on your setup.

In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these paths for your setup
# ============================================================================

# Dataset selection: 'celeba' or 'celebahq'
DATASET = 'celeba'  # Change to 'celebahq' for CelebA-HQ

# Image size for VAE and index building
IMAGE_SIZE = 128
LATENT_DIM = 256

# Smoothing parameters
SIGMA = 0.25  # noise scale
KNN_K = 64    # neighbors for manifold smoothing
N_SAMPLES = 100  # Monte Carlo samples for certification

# Paths
if DATASET == 'celeba':
    DATA_ROOT = Path("/BS/databases08/CelebA")
    IMAGE_DIR = DATA_ROOT / "img_align_celeba"
    ANNO_FILE = DATA_ROOT / "Anno" / "list_attr_celeba.txt"
    VAE_CKPT = repo_root / "output" / "pretrained_model" / "vae_celeba_128" / "best.pt"
    CLASSIFIER_CKPT = repo_root / "output" / "pretrained_model" / "smile_resnet_celeba" / "best.pt"
    FILE_EXT = ""
else:
    DATA_ROOT = Path("/BS/databases08/CelebA-HQ/data512x512")
    IMAGE_DIR = DATA_ROOT / "train"
    ANNO_FILE = repo_root / "input" / "datasets" / "celebAHQ" / "Anno" / "list_attr_celeba_hq.txt"
    VAE_CKPT = repo_root / "output" / "pretrained_model" / "vae_celebahq_128" / "best.pt"
    CLASSIFIER_CKPT = repo_root / "output" / "pretrained_model" / "smile_resnet_celebahq" / "best.pt"
    FILE_EXT = ".jpg"

# Index paths
INDEX_DIR = repo_root / "notebooks" / "indices"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
PIXEL_INDEX_PATH = INDEX_DIR / f"{DATASET}_pixel_{IMAGE_SIZE}.ann"
LATENT_INDEX_PATH = INDEX_DIR / f"{DATASET}_latent_{LATENT_DIM}.ann"

print(f"Dataset: {DATASET}")
print(f"Image dir: {IMAGE_DIR}")
print(f"VAE checkpoint: {VAE_CKPT}")
print(f"Classifier checkpoint: {CLASSIFIER_CKPT}")

## Load Dataset

In [ ]:
def read_celeba_annotations(annotation_path: Path):
    """Parse CelebA-style attribute file for Smiling attribute."""
    lines = [line.strip() for line in annotation_path.read_text().splitlines() if line.strip()]
    if len(lines) < 3:
        raise ValueError(f"Invalid CelebA attribute file: {annotation_path}")
    
    attr_names = lines[1].split()
    if "Smiling" not in attr_names:
        raise ValueError(f"Smiling attribute not found in {annotation_path}")
    smile_idx = attr_names.index("Smiling")
    
    labels = {}
    for row in lines[2:]:
        parts = row.split()
        if len(parts) < 2 + smile_idx:
            continue
        file_name = parts[0]
        val = int(parts[1 + smile_idx])
        labels[file_name] = 1 if val > 0 else 0
    return labels


# Load annotations
labels = read_celeba_annotations(ANNO_FILE)
print(f"Total annotations: {len(labels)}")
print(f"Smiling: {sum(labels.values())}, Not smiling: {len(labels) - sum(labels.values())}")

# Build sample list
samples = []
for fname, label in labels.items():
    if FILE_EXT and not fname.endswith(FILE_EXT):
        path = IMAGE_DIR / f"{fname}{FILE_EXT}"
    else:
        path = IMAGE_DIR / fname
    if path.exists():
        samples.append((path, label))

samples.sort(key=lambda x: x[0].name)
print(f"Found {len(samples)} images")

# Use subset for notebook demo
SUBSET_SIZE = 500
if len(samples) > SUBSET_SIZE:
    random.shuffle(samples)
    samples = samples[:SUBSET_SIZE]
    samples.sort(key=lambda x: x[0].name)
    print(f"Using subset of {SUBSET_SIZE} samples")

## Load Models

In [ ]:
# Transforms
CELEBA_MEAN = [0.5, 0.5, 0.5]
CELEBA_STD = [0.5, 0.5, 0.5]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transform for VAE/smoothing
vae_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CELEBA_MEAN, CELEBA_STD),
])

# Transform for classifier
classifier_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load VAE
vae = ConvVAE(in_channels=3, image_size=IMAGE_SIZE, latent_dim=LATENT_DIM).to(device)
if VAE_CKPT.exists():
    load_vae_checkpoint(vae, str(VAE_CKPT), device)
    vae.eval()
    print(f"VAE loaded from {VAE_CKPT}")
else:
    print(f"WARNING: VAE checkpoint not found at {VAE_CKPT}")
    print("Latent smoothing will not be available. Train VAE first.")

# Load classifier
classifier = build_resnet_classifier(name="resnet18", pretrained=False, num_classes=1).to(device)
if CLASSIFIER_CKPT.exists():
    ckpt = torch.load(CLASSIFIER_CKPT, map_location=device)
    if "model_state_dict" in ckpt:
        classifier.load_state_dict(ckpt["model_state_dict"])
    else:
        classifier.load_state_dict(ckpt)
    classifier.eval()
    print(f"Classifier loaded from {CLASSIFIER_CKPT}")
else:
    print(f"WARNING: Classifier checkpoint not found at {CLASSIFIER_CKPT}")

## Build Annoy Indices

Build kNN indices for both pixel space and latent space.

In [ ]:
def build_pixel_index(samples, image_size, index_path, n_trees=50):
    """Build Annoy index over flattened pixel vectors."""
    dim = 3 * image_size * image_size
    ann_index = annoy.AnnoyIndex(dim, 'euclidean')
    index_map = {}
    
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(CELEBA_MEAN, CELEBA_STD),
    ])
    
    for idx, (path, label) in enumerate(tqdm(samples, desc="Building pixel index")):
        img = Image.open(path).convert("RGB")
        img_tensor = transform(img)
        flat = img_tensor.numpy().flatten().astype(np.float32)
        ann_index.add_item(idx, flat)
        index_map[idx] = str(path)
    
    ann_index.build(n_trees)
    ann_index.save(str(index_path))
    
    map_path = index_path.with_suffix(".json")
    with open(map_path, "w") as f:
        json.dump(index_map, f)
    
    print(f"Pixel index saved: {index_path} ({len(samples)} items)")
    return ann_index


def build_latent_index(samples, vae, index_path, n_trees=50):
    """Build Annoy index over VAE latent vectors."""
    dim = vae.latent_dim
    ann_index = annoy.AnnoyIndex(dim, 'euclidean')
    index_map = {}
    
    transform = transforms.Compose([
        transforms.Resize((vae.image_size, vae.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(CELEBA_MEAN, CELEBA_STD),
    ])
    
    vae.eval()
    with torch.no_grad():
        for idx, (path, label) in enumerate(tqdm(samples, desc="Building latent index")):
            img = Image.open(path).convert("RGB")
            img_tensor = transform(img).unsqueeze(0).to(device)
            mu, _ = vae.encode(img_tensor)
            z = mu.squeeze(0).cpu().numpy().astype(np.float32)
            ann_index.add_item(idx, z)
            index_map[idx] = str(path)
    
    ann_index.build(n_trees)
    ann_index.save(str(index_path))
    
    map_path = index_path.with_suffix(".json")
    with open(map_path, "w") as f:
        json.dump(index_map, f)
    
    print(f"Latent index saved: {index_path} ({len(samples)} items)")
    return ann_index


# Build or load pixel index
if PIXEL_INDEX_PATH.exists():
    pixel_index = annoy.AnnoyIndex(3 * IMAGE_SIZE * IMAGE_SIZE, 'euclidean')
    pixel_index.load(str(PIXEL_INDEX_PATH))
    print(f"Loaded pixel index: {PIXEL_INDEX_PATH}")
else:
    pixel_index = build_pixel_index(samples, IMAGE_SIZE, PIXEL_INDEX_PATH)

# Build or load latent index
if LATENT_INDEX_PATH.exists():
    latent_index = annoy.AnnoyIndex(LATENT_DIM, 'euclidean')
    latent_index.load(str(LATENT_INDEX_PATH))
    print(f"Loaded latent index: {LATENT_INDEX_PATH}")
elif VAE_CKPT.exists():
    latent_index = build_latent_index(samples, vae, LATENT_INDEX_PATH)
else:
    latent_index = None
    print("Skipping latent index (VAE not available)")

## Smoothing Functions

In [ ]:
def unnormalize(img_tensor, mean=CELEBA_MEAN, std=CELEBA_STD):
    """Convert normalized tensor back to [0,1] range for display."""
    img = img_tensor.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1)


def get_neighbors(index, item_idx, k):
    """Get neighbor vectors from Annoy index."""
    nn_idxs = index.get_nns_by_item(item_idx, k)
    neighbors = np.array([index.get_item_vector(i) for i in nn_idxs], dtype=np.float32)
    return neighbors


def sample_pixel_isotropic(img_tensor, sigma):
    """Add isotropic Gaussian noise."""
    return img_tensor + torch.randn_like(img_tensor) * sigma


def sample_pixel_manifold(img_tensor, pixel_index, item_idx, sigma, knn_k, eps_eig=1e-6):
    """Sample from local PCA manifold in pixel space."""
    flat = img_tensor.numpy().flatten()
    neighbors = get_neighbors(pixel_index, item_idx, knn_k)
    pca = fit_local_pca(neighbors, eps_eig=eps_eig)
    white = whiten(flat, pca)
    white_noised = white + np.random.randn(len(white)).astype(np.float32) * sigma
    unwhite = unwhiten(white_noised, pca)
    return torch.from_numpy(unwhite.reshape(img_tensor.shape)).float()


def sample_latent_isotropic(img_tensor, vae, sigma, device):
    """Sample by adding isotropic noise in latent space."""
    with torch.no_grad():
        x = img_tensor.unsqueeze(0).to(device)
        mu, _ = vae.encode(x)
        z_noised = mu + torch.randn_like(mu) * sigma
        x_hat = vae.decode(z_noised)
    return x_hat.squeeze(0).cpu()


def sample_latent_manifold(img_tensor, vae, latent_index, item_idx, sigma, knn_k, device, eps_eig=1e-6):
    """Sample from local PCA manifold in latent space."""
    with torch.no_grad():
        x = img_tensor.unsqueeze(0).to(device)
        mu, _ = vae.encode(x)
        z = mu.squeeze(0).cpu().numpy()
    
    neighbors = get_neighbors(latent_index, item_idx, knn_k)
    pca = fit_local_pca(neighbors, eps_eig=eps_eig)
    white = whiten(z, pca)
    white_noised = white + np.random.randn(len(white)).astype(np.float32) * sigma
    z_noised = unwhiten(white_noised, pca)
    
    with torch.no_grad():
        z_t = torch.from_numpy(z_noised[None, :]).to(device=device, dtype=torch.float32)
        x_hat = vae.decode(z_t)
    return x_hat.squeeze(0).cpu()

## Visualization: Smoothed Samples

Compare pixel vs latent smoothing visually.

In [ ]:
def visualize_smoothing_comparison(sample_idx, n_samples=5):
    """Visualize original, neighbors, and smoothed samples for different methods."""
    path, label = samples[sample_idx]
    img = Image.open(path).convert("RGB")
    img_tensor = vae_transform(img)
    
    fig, axes = plt.subplots(4, n_samples + 2, figsize=(3*(n_samples+2), 12))
    
    # Row 0: Original + Neighbors
    axes[0, 0].imshow(unnormalize(img_tensor).permute(1, 2, 0).numpy())
    axes[0, 0].set_title(f"Original\n{'Smile' if label else 'No Smile'}")
    axes[0, 0].axis('off')
    
    # Show pixel neighbors
    nn_idxs = pixel_index.get_nns_by_item(sample_idx, n_samples + 1)[1:]  # skip self
    for i, nn_idx in enumerate(nn_idxs[:n_samples]):
        nn_path, nn_label = samples[nn_idx]
        nn_img = Image.open(nn_path).convert("RGB")
        nn_tensor = vae_transform(nn_img)
        axes[0, i+1].imshow(unnormalize(nn_tensor).permute(1, 2, 0).numpy())
        axes[0, i+1].set_title(f"NN {i+1}")
        axes[0, i+1].axis('off')
    axes[0, -1].axis('off')
    axes[0, 0].set_ylabel("Pixel Neighbors", fontsize=12)
    
    # Row 1: Pixel Isotropic
    axes[1, 0].imshow(unnormalize(img_tensor).permute(1, 2, 0).numpy())
    axes[1, 0].set_title("Original")
    axes[1, 0].axis('off')
    for i in range(n_samples):
        noised = sample_pixel_isotropic(img_tensor, SIGMA)
        axes[1, i+1].imshow(unnormalize(noised).permute(1, 2, 0).numpy().clip(0, 1))
        axes[1, i+1].set_title(f"Sample {i+1}")
        axes[1, i+1].axis('off')
    axes[1, -1].axis('off')
    axes[1, 0].set_ylabel("Pixel Isotropic", fontsize=12)
    
    # Row 2: Pixel Manifold
    axes[2, 0].imshow(unnormalize(img_tensor).permute(1, 2, 0).numpy())
    axes[2, 0].set_title("Original")
    axes[2, 0].axis('off')
    for i in range(n_samples):
        noised = sample_pixel_manifold(img_tensor, pixel_index, sample_idx, SIGMA, KNN_K)
        axes[2, i+1].imshow(unnormalize(noised).permute(1, 2, 0).numpy().clip(0, 1))
        axes[2, i+1].set_title(f"Sample {i+1}")
        axes[2, i+1].axis('off')
    axes[2, -1].axis('off')
    axes[2, 0].set_ylabel("Pixel Manifold", fontsize=12)
    
    # Row 3: Latent Manifold (if available)
    if latent_index is not None:
        axes[3, 0].imshow(unnormalize(img_tensor).permute(1, 2, 0).numpy())
        axes[3, 0].set_title("Original")
        axes[3, 0].axis('off')
        for i in range(n_samples):
            noised = sample_latent_manifold(img_tensor, vae, latent_index, sample_idx, SIGMA*2, KNN_K, device)
            axes[3, i+1].imshow(unnormalize(noised).permute(1, 2, 0).numpy().clip(0, 1))
            axes[3, i+1].set_title(f"Sample {i+1}")
            axes[3, i+1].axis('off')
        axes[3, -1].axis('off')
        axes[3, 0].set_ylabel("Latent Manifold", fontsize=12)
    else:
        for ax in axes[3]:
            ax.axis('off')
        axes[3, 0].text(0.5, 0.5, "VAE not available", ha='center', va='center')
    
    plt.suptitle(f"Smoothing Comparison - Sample {sample_idx}", fontsize=14)
    plt.tight_layout()
    plt.show()


# Visualize a few samples
for idx in [0, 10, 50]:
    if idx < len(samples):
        visualize_smoothing_comparison(idx)

## Run Certification

In [ ]:
@torch.no_grad()
def certify_sample(sample_idx, smoothing_mode='pixel_manifold', n_samples=N_SAMPLES):
    """Certify a single sample using randomized smoothing.
    
    Args:
        smoothing_mode: 'pixel_isotropic', 'pixel_manifold', 'latent_isotropic', 'latent_manifold'
    """
    path, label = samples[sample_idx]
    img = Image.open(path).convert("RGB")
    img_tensor = vae_transform(img)
    
    class_counts = np.zeros(2, dtype=np.int64)
    classifier.eval()
    
    for _ in range(n_samples):
        # Generate smoothed sample
        if smoothing_mode == 'pixel_isotropic':
            noised = sample_pixel_isotropic(img_tensor, SIGMA)
        elif smoothing_mode == 'pixel_manifold':
            noised = sample_pixel_manifold(img_tensor, pixel_index, sample_idx, SIGMA, KNN_K)
        elif smoothing_mode == 'latent_isotropic':
            noised = sample_latent_isotropic(img_tensor, vae, SIGMA, device)
        elif smoothing_mode == 'latent_manifold':
            noised = sample_latent_manifold(img_tensor, vae, latent_index, sample_idx, SIGMA*2, KNN_K, device)
        else:
            raise ValueError(f"Unknown smoothing mode: {smoothing_mode}")
        
        # Convert for classifier
        noised_pil = transforms.ToPILImage()(unnormalize(noised))
        x = classifier_transform(noised_pil).unsqueeze(0).to(device)
        
        logit = classifier(x).squeeze()
        pred = 1 if torch.sigmoid(logit).item() > 0.5 else 0
        class_counts[pred] += 1
    
    cert = certify_token_from_counts(
        class_counts,
        alpha_noise=SIGMA,
        alpha_conf=0.001,
        abstain_label=-1,
    )
    
    return {
        'idx': sample_idx,
        'label': label,
        'pred': cert.pred,
        'radius': cert.radius,
        'abstained': cert.abstained,
        'correct': (cert.pred == label) if not cert.abstained else False,
        'class_counts': class_counts.tolist(),
    }


# Test on a few samples
print("Testing certification on sample 0:")
result = certify_sample(0, smoothing_mode='pixel_manifold')
print(f"  Label: {result['label']}, Pred: {result['pred']}, Radius: {result['radius']:.4f}, Abstained: {result['abstained']}")

In [ ]:
def run_certification_experiment(smoothing_mode, num_samples=100):
    """Run certification on multiple samples and compute metrics."""
    results = []
    
    test_indices = list(range(min(num_samples, len(samples))))
    
    for idx in tqdm(test_indices, desc=f"Certifying ({smoothing_mode})"):
        result = certify_sample(idx, smoothing_mode=smoothing_mode)
        results.append(result)
    
    # Compute metrics
    total = len(results)
    certified = [r for r in results if not r['abstained']]
    correct = [r for r in results if r['correct']]
    radii = [r['radius'] for r in certified]
    
    metrics = {
        'smoothing_mode': smoothing_mode,
        'total_samples': total,
        'certified_samples': len(certified),
        'abstain_rate': (total - len(certified)) / total,
        'certified_accuracy': len(correct) / total,
        'mean_radius': np.mean(radii) if radii else 0.0,
        'median_radius': np.median(radii) if radii else 0.0,
    }
    
    # Class-wise
    for cls in [0, 1]:
        cls_results = [r for r in results if r['label'] == cls]
        cls_correct = sum(1 for r in cls_results if r['correct'])
        metrics[f'class_{cls}_accuracy'] = cls_correct / len(cls_results) if cls_results else 0.0
    
    return metrics, results


# Run experiments
NUM_TEST = 100  # number of samples to certify

experiments = {}

# Pixel isotropic
print("\n" + "="*60)
print("Running Pixel Isotropic Certification")
print("="*60)
experiments['pixel_isotropic'] = run_certification_experiment('pixel_isotropic', NUM_TEST)

# Pixel manifold
print("\n" + "="*60)
print("Running Pixel Manifold Certification")
print("="*60)
experiments['pixel_manifold'] = run_certification_experiment('pixel_manifold', NUM_TEST)

# Latent manifold (if available)
if latent_index is not None:
    print("\n" + "="*60)
    print("Running Latent Manifold Certification")
    print("="*60)
    experiments['latent_manifold'] = run_certification_experiment('latent_manifold', NUM_TEST)

## Results Comparison

In [ ]:
# Print comparison table
print("\n" + "="*80)
print(f"CERTIFICATION RESULTS COMPARISON - {DATASET.upper()}")
print("="*80)
print(f"{'Method':<20} {'Certified Acc':>15} {'Abstain Rate':>15} {'Mean Radius':>15} {'Class 0 Acc':>12} {'Class 1 Acc':>12}")
print("-"*80)

for name, (metrics, results) in experiments.items():
    print(f"{name:<20} {metrics['certified_accuracy']*100:>14.2f}% {metrics['abstain_rate']*100:>14.2f}% {metrics['mean_radius']:>15.4f} {metrics['class_0_accuracy']*100:>11.2f}% {metrics['class_1_accuracy']*100:>11.2f}%")

print("="*80)

In [ ]:
# Visualization: Radius distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of certified accuracy
methods = list(experiments.keys())
accuracies = [experiments[m][0]['certified_accuracy'] * 100 for m in methods]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(methods)))

axes[0].bar(methods, accuracies, color=colors)
axes[0].set_ylabel('Certified Accuracy (%)')
axes[0].set_title(f'Certified Accuracy Comparison - {DATASET}')
axes[0].set_ylim(0, 100)
for i, (m, acc) in enumerate(zip(methods, accuracies)):
    axes[0].text(i, acc + 1, f'{acc:.1f}%', ha='center')

# Radius histogram
for i, (name, (metrics, results)) in enumerate(experiments.items()):
    radii = [r['radius'] for r in results if not r['abstained']]
    if radii:
        axes[1].hist(radii, bins=20, alpha=0.5, label=name, color=colors[i])

axes[1].set_xlabel('Certified Radius')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Certified Radii')
axes[1].legend()

plt.tight_layout()
plt.savefig(INDEX_DIR / f'{DATASET}_certification_comparison.png', dpi=150)
plt.show()

print(f"Figure saved: {INDEX_DIR / f'{DATASET}_certification_comparison.png'}")

## Save Results

In [ ]:
# Save all results
results_dir = repo_root / "output" / "certification" / f"{DATASET}_notebook"
results_dir.mkdir(parents=True, exist_ok=True)

# Save metrics summary
all_metrics = {name: metrics for name, (metrics, _) in experiments.items()}
with open(results_dir / "metrics_summary.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

# Save per-sample results for each method
import csv
for name, (metrics, results) in experiments.items():
    csv_path = results_dir / f"{name}_results.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=['idx', 'label', 'pred', 'radius', 'abstained', 'correct'])
        writer.writeheader()
        for r in results:
            writer.writerow({k: r[k] for k in ['idx', 'label', 'pred', 'radius', 'abstained', 'correct']})
    print(f"Saved: {csv_path}")

print(f"\nAll results saved to: {results_dir}")

## Summary

This notebook demonstrated:

1. **Index Building**: Created Annoy indices for both pixel-space (flattened images) and latent-space (VAE embeddings)

2. **Smoothing Methods**:
   - **Pixel Isotropic**: Standard Gaussian noise in image space
   - **Pixel Manifold**: Local PCA-based smoothing using kNN in pixel space
   - **Latent Manifold**: Local PCA-based smoothing in VAE latent space

3. **Certification**: Used randomized smoothing to certify predictions with provable robustness guarantees

**Key Observations**:
- Manifold smoothing typically produces more semantically meaningful perturbations
- Latent space smoothing preserves high-level features better than pixel-space
- The certified radius depends on both sigma and the consistency of predictions

**Next Steps**:
- Compare CelebA vs CelebA-HQ results
- Tune sigma and knn_k for optimal certified accuracy
- Run full-scale experiments using the command-line workflow